In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS oil_stock")
spark.sql("USE CATALOG oil_stock")


In [0]:
bronze_df = spark.table("oil_stock.bronze.market_daily")

display(bronze_df)

In [0]:
from pyspark.sql import functions as F

silver_daily_df = (
    bronze_df
    .select(
        F.col('trading_date'),
        F.col('symbol'),
        F.col('currency'),
        F.col('close').cast('double'),
        F.col('high').cast('double'),
        F.col('low').cast('double'),
        F.col('open').cast('double'),
        F.col('volume').cast('long')
    )
    .dropDuplicates(['trading_date', 'symbol'])
    .filter(F.col('close').isNotNull())
)

display(silver_daily_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

silver_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.market_daily")